In [4]:
import torch
import timm
from tqdm import tqdm
from aimet_torch.batch_norm_fold import fold_all_batch_norms
from torch.utils.data import DataLoader
import aimet_torch.quantsim as qsim
from torchvision import transforms, datasets
from aimet_torch.v2.nn import QuantizationMixin
from timm.layers.drop import DropPath
from timm.layers.adaptive_avgmax_pool import FastAdaptiveAvgPool

model = timm.create_model("swin_tiny_patch4_window7_224", pretrained=True)
# model.load_state_dict(torch.load("swin/Swin-Transformer/checkpoints/swin_tiny_patch4_window7_224.pth"))

model.eval()


# Validation function for PyTorch model with progress bar
def validate_pytorch(model, dataloader, device):
    model.eval()
    top1_correct = 0
    top5_correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Validating PyTorch Model", leave=False):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, pred_top1 = outputs.topk(1, dim=1)
            _, pred_top5 = outputs.topk(5, dim=1)

            top1_correct += (pred_top1.squeeze() == labels).sum().item()
            top5_correct += sum([labels[i] in pred_top5[i] for i in range(len(labels))])
            total += labels.size(0)

    top1_acc = 100 * top1_correct / total
    top5_acc = 100 * top5_correct / total
    return top1_acc, top5_acc

# Define ImageNet validation dataset and DataLoader
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

dataset = datasets.ImageFolder("/media/bmw/datasets/imagenet-1k/val", transform=transform)
dataloader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Validate model
# top1_acc, top5_acc = validate_pytorch(model, dataloader, device)
# print(f"Top-1 Accuracy: {top1_acc:.2f}%")
# print(f"Top-5 Accuracy: {top5_acc:.2f}%")


2025-02-27 11:16:27,286 - timm.models._builder - INFO - Loading pretrained weights from Hugging Face hub (timm/swin_tiny_patch4_window7_224.ms_in1k)
2025-02-27 11:16:27,570 - timm.models._hub - INFO - [timm/swin_tiny_patch4_window7_224.ms_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.


In [ ]:
from aimet_torch.model_validator.model_validator import ModelValidator
# Check if model is valid for AIMET
invalid_layers = ModelValidator.validate_model(model, model_input=torch.randn(1, 3, 224, 224))

if invalid_layers:
    print("\n❌ Model contains unsupported layers for AIMET quantization:")
    for layer, issue in invalid_layers.items():
        print(f"- {layer}: {issue}")
else:
    print("\n✅ Model is valid for AIMET quantization!")

# Perform Batch Norm Folding
fold_all_batch_norms(model, input_shapes=(1, 3, 224, 224))

# Validate model after BN folding
print("\nValidating model after BatchNorm folding...")
top1_acc_bn, top5_acc_bn = validate_pytorch(model, dataloader, device)
print(f"BN-Folded Model - Top-1 Accuracy: {top1_acc_bn:.2f}% | Top-5 Accuracy: {top5_acc_bn:.2f}%")

# Check model again after BN folding
invalid_layers_after_bn = ModelValidator.validate_model(model, model_input=torch.randn(1, 3, 224, 224))

if invalid_layers_after_bn:
    print("\n❌ Model still has unsupported layers after BN folding:")
    for layer, issue in invalid_layers_after_bn.items():
        print(f"- {layer}: {issue}")
else:
    print("\n✅ Model is valid for AIMET quantization after BN folding!")


2025-02-27 11:16:58,175 - Utils - INFO - Running validator check <function validate_for_reused_modules at 0x7f211e72b400>
2025-02-27 11:16:59,047 - Utils - INFO - Running validator check <function validate_for_missing_modules at 0x7f211e72a5f0>


/media/bmw/harisharajan_r_r/.conda/lib/python3.10/site-packages/torch/__init__.py:1404: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  assert condition, message


2025-02-27 11:17:00,486 - ConnectedGraph - WARNING - Unable to isolate model outputs.
2025-02-27 11:17:00,516 - Utils - ERROR - Functional ops that are not marked as math invariant were found in the model. AIMET features will not work properly for such ops.
Consider the following choices: 
1. Redefine as a torch.nn.Module in the class definition.
2. The op can remain as a functional op due to being math invariant, but the op type has not been added to ConnectedGraph.math_invariant_types set. 
Add an entry to ignore the op by importing the set and adding the op type:

	from aimet_torch.meta.connectedgraph import ConnectedGraph
	ConnectedGraph.math_invariant_types.add(...)

The following functional ops were found. The parent module is named for ease of locating the ops within the model definition.
	remainder_42; op_type: remainder                  parent module: SwinTransformer.layers.0.blocks.0
	rsub_45; op_type: rsub                            parent module: SwinTransformer.layers.0.bl

Validating PyTorch Model:  41%|████▏     | 324/782 [18:31<35:06,  4.60s/it]

In [ ]:

@QuantizationMixin.implements(DropPath)
class QuantizedDropPath(QuantizationMixin, DropPath):
    def __quant_init__(self):
        super().__quant_init__()

        # Declare the number of input/output quantizers
        self.input_quantizers = torch.nn.ModuleList([None])
        self.output_quantizers = torch.nn.ModuleList([None])

    def forward(self, x):
        # Quantize input tensors
        if self.input_quantizers[0]:
            x = self.input_quantizers[0](x)

        # Run forward with quantized inputs and parameters
        with self._patch_quantized_parameters():
            ret = super().forward(x)

        # Quantize output tensors
        if self.output_quantizers[0]:
            ret = self.output_quantizers[0](ret)

        return ret
    


@QuantizationMixin.implements(FastAdaptiveAvgPool)
class QuantizedFastAdaptiveAvgPool(QuantizationMixin, FastAdaptiveAvgPool):
    def __quant_init__(self):
        super().__quant_init__()

        # Declare the number of input/output quantizers
        self.input_quantizers = torch.nn.ModuleList([None])
        self.output_quantizers = torch.nn.ModuleList([None])

    def forward(self, x):
        # Quantize input tensors
        if self.input_quantizers[0]:
            x = self.input_quantizers[0](x)

        # Run forward with quantized inputs and parameters
        with self._patch_quantized_parameters():
            ret = super().forward(x)

        # Quantize output tensors
        if self.output_quantizers[0]:
            ret = self.output_quantizers[0](ret)

        return ret


In [ ]:

def apply_aimet_quantization(model, device, dataloader):
    dummy_input = torch.rand(1, 3, 224, 224).to(device)
    
    # Print model before BN folding
    print("\nModel before BatchNorm folding:")
    print(model)
    
    fold_all_batch_norms(model, input_shapes=(1, 3, 224, 224), dummy_input=dummy_input)
    
    # Print model after BN folding
    print("\nModel after BatchNorm folding:")
    print(model)
    
    quant_sim = qsim.QuantizationSimModel( 
        model, 
        dummy_input=dummy_input, 
        quant_scheme= 'tf_enhanced',
        rounding_mode= 'nearest',
        default_param_bw= 8,
        default_output_bw = 8
    )
    
    # Use real calibration data for encoding computation
    def forward_pass(model, dataloader):
        model.eval()
        with torch.no_grad():
            for i, (images, _) in enumerate(dataloader):
                print(i , images[0].shape)
                if i >= 10:  # Use only first 10 batches for calibration
                    
                    break
                model(images.to(device))
                # print()
    
    quant_sim.compute_encodings(forward_pass, dataloader)


    return quant_sim.model

quantized_model = apply_aimet_quantization(model,  device, dataloader)

# Validate model
top1_acc, top5_acc = validate_pytorch(quantized_model, dataloader, device)
print(f"Top-1 Accuracy: {top1_acc:.2f}%")
print(f"Top-5 Accuracy: {top5_acc:.2f}%")
